# Chapitre 4 — Partie 2 : Métriques de Classification

**Durée estimée : 2h30**

## 🎯 Objectifs d'apprentissage

À la fin de cette partie, vous serez capable de :
1. **Analyser** une matrice de confusion pour diagnostiquer les erreurs d'un modèle
2. **Calculer** et interpréter Precision, Recall et F1-Score
3. **Tracer** et interpréter une courbe ROC-AUC
4. **Choisir** la métrique appropriée selon le contexte business

---

## Lien avec les chapitres précédents

Dans le **Chapitre 2 (Leçon 5)**, nous avons découvert la matrice de confusion et les métriques precision/recall. Nous allons maintenant les approfondir et ajouter de nouvelles métriques essentielles : le F1-Score et la courbe ROC-AUC.

Rappelons que dans le **Chapitre 2**, nous avons vu l'importance de comparer notre modèle à une baseline (DummyClassifier). Ici, nous allons aller plus loin dans l'analyse des erreurs.

---

## 🌍 Problème Réel : Le paradoxe du modèle à 99% d'accuracy

Vous travaillez pour une banque. Votre mission : détecter les transactions frauduleuses.

Après des semaines de travail, vous présentez fièrement votre modèle : **99% d'accuracy !**

Votre manager vous félicite... jusqu'à ce qu'il remarque un détail troublant :

**"Combien de fraudes avez-vous détectées ?"**

Vous vérifiez : **zéro fraude détectée**.

---

### Comment est-ce possible ?

Sur 10 000 transactions :
- 9 900 sont légitimes (99%)
- 100 sont frauduleuses (1%)

Un modèle qui prédit **"légitime"** pour TOUT le monde aura :
- 9 900 bonnes prédictions (légitimes)
- 100 mauvaises prédictions (fraudes manquées)
- **Accuracy = 99%** ✓

Mais ce modèle est **complètement inutile** pour son objectif !

> Selon une [analyse d'Arize AI](https://arize.com/blog-course/f1-score/), dans les cas de détection de fraude où 99% des transactions sont légitimes, l'accuracy seule est trompeuse. Un modèle qui dit toujours "pas de fraude" atteint 99% d'accuracy mais ne détecte aucune fraude.

**Question :** Que faudrait-il mesurer pour vraiment évaluer ce modèle de détection de fraude ?

*(Réponse attendue : Il faut mesurer combien de fraudes on a réellement détectées parmi toutes les vraies fraudes, et combien de nos alertes "fraude" sont vraiment des fraudes.)*

---

C'est exactement ce que mesurent **Precision** et **Recall** !

---

## 2.1 La Matrice de Confusion : Anatomie des Erreurs

Comme vu au Chapitre 2, la matrice de confusion décompose les prédictions en 4 catégories.

```
┌─────────────────────────────────────────────────────────────────────┐
│              MATRICE DE CONFUSION (Classification Binaire)         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│                           PRÉDICTION                                │
│                      Négatif       Positif                          │
│                   ┌───────────┬───────────┐                         │
│         Négatif   │    TN     │    FP     │  ← Vrais Négatifs       │
│  RÉALITÉ          │  (Vrai    │ (Faux     │                         │
│                   │  Négatif) │  Positif) │     (Fausse alerte)     │
│                   ├───────────┼───────────┤                         │
│         Positif   │    FN     │    TP     │  ← Vrais Positifs       │
│                   │  (Faux    │  (Vrai    │                         │
│                   │  Négatif) │  Positif) │     (Cas manqué !)      │
│                   └───────────┴───────────┘                         │
│                                                                     │
│   TN = Transaction légitime ✓ → Prédite légitime ✓                 │
│   FP = Transaction légitime ✓ → Prédite fraude ✗ (client embêté)   │
│   FN = Vraie fraude ✗ → Prédite légitime ✗ (fraude manquée !)      │
│   TP = Vraie fraude ✗ → Prédite fraude ✓ (fraude détectée !)       │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

**Question :** Dans la détection de fraude, quelle erreur est la plus grave : FP ou FN ?

*(Réponse attendue : FN (Faux Négatif) = fraude manquée. Une fraude non détectée cause une perte financière. Un FP (fausse alerte) n'est qu'un désagrément pour le client.)*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Exemple : détection de fraude
# 100 transactions : 90 légitimes, 10 fraudes
np.random.seed(42)

# Vraies valeurs (0 = légitime, 1 = fraude)
y_vrai = np.array([0]*90 + [1]*10)

# Prédictions d'un modèle imparfait
# Le modèle détecte 7 fraudes sur 10, mais génère aussi 5 fausses alertes
y_pred = np.array([0]*85 + [1]*5 + [0]*3 + [1]*7)

print("📊 Exemple : Détection de fraude")
print("=" * 50)
print(f"Transactions totales : {len(y_vrai)}")
print(f"  - Légitimes : {sum(y_vrai == 0)}")
print(f"  - Fraudes : {sum(y_vrai == 1)}")

In [ ]:
# Calculer la matrice de confusion
cm = confusion_matrix(y_vrai, y_pred)

# Extraire les valeurs
TN, FP, FN, TP = cm.ravel()

print("\n📊 Matrice de Confusion")
print("=" * 50)
print(f"\nVrais Négatifs  (TN) : {TN} ← Légitimes correctement identifiées")
print(f"Faux Positifs   (FP) : {FP} ← Fausses alertes (clients embêtés)")
print(f"Faux Négatifs   (FN) : {FN} ← Fraudes MANQUÉES ! ⚠️")
print(f"Vrais Positifs  (TP) : {TP} ← Fraudes détectées ✓")

In [ ]:
# Visualiser
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Légitime', 'Fraude'])
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Matrice de Confusion - Détection de Fraude', fontsize=14)
plt.tight_layout()
plt.show()

---

## 2.2 Accuracy : La Métrique Trompeuse

L'accuracy mesure simplement le pourcentage de prédictions correctes :

$$Accuracy = \frac{TN + TP}{TN + FP + FN + TP}$$

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_vrai, y_pred)
accuracy_manuel = (TN + TP) / (TN + FP + FN + TP)

print(f"Accuracy = ({TN} + {TP}) / ({TN} + {FP} + {FN} + {TP})")
print(f"Accuracy = {accuracy:.1%}")
print(f"\n⚠️ 92% semble bon... mais on a manqué {FN} fraudes sur 10 !")

<details>
<summary>🤔 Question Socratique : Quand l'accuracy est-elle une bonne métrique ?</summary>

### 🔑 Réponse

L'accuracy est appropriée quand :

1. **Les classes sont équilibrées** (~50/50)
2. **Toutes les erreurs ont le même coût** (FP et FN sont également graves)
3. **Vous avez suffisamment de données** dans chaque classe

**Exemples où l'accuracy fonctionne bien :**
- Classification d'images (chat vs chien avec ~50% chaque)
- Analyse de sentiments équilibrée (positif/négatif)

**Exemples où l'accuracy est trompeuse :**
- Détection de fraude (1% de fraudes)
- Diagnostic de maladies rares (0.1% de cas)
- Détection de défauts industriels (2% de défauts)

</details>

---

## 2.3 Precision : "Parmi mes alertes, combien sont correctes ?"

### Construire l'intuition

Imaginez que vous êtes le responsable qui reçoit les alertes de fraude. Chaque alerte nécessite une investigation coûteuse.

**Question :** Si le modèle vous envoie 12 alertes par jour, combien sont de vraies fraudes ?

C'est exactement ce que mesure la **Precision** :

```
┌─────────────────────────────────────────────────────────────────────┐
│              PRECISION : La fiabilité des alertes                   │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│                     Prédictions "FRAUDE"                            │
│                   ┌─────────────────────────┐                       │
│                   │  TP = 7    │  FP = 5    │                       │
│                   │ (vraies    │ (fausses   │                       │
│                   │  fraudes)  │  alertes)  │                       │
│                   └─────────────────────────┘                       │
│                            Total = 12                               │
│                                                                     │
│   Precision = TP / (TP + FP) = 7 / 12 = 58.3%                      │
│                                                                     │
│   → "58% de mes alertes sont de vraies fraudes"                    │
│   → "42% de mes alertes sont des fausses alertes"                  │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
from sklearn.metrics import precision_score

precision = precision_score(y_vrai, y_pred)
precision_manuel = TP / (TP + FP)

print("📊 PRECISION")
print("=" * 50)
print(f"\nFormule : TP / (TP + FP)")
print(f"        = {TP} / ({TP} + {FP})")
print(f"        = {precision:.1%}")
print(f"\n💡 Interprétation :")
print(f"   Sur 12 alertes de fraude, {TP} sont de vraies fraudes.")
print(f"   → {precision:.1%} de fiabilité des alertes")

---

┌─────────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : Precision (Précision)                               │
│                                                                     │
│ La Precision mesure la proportion de prédictions positives qui     │
│ sont réellement positives.                                          │
│                                                                     │
│ Formule : Precision = TP / (TP + FP)                               │
│                                                                     │
│ Question qu'elle répond :                                           │
│ "Parmi tout ce que j'ai prédit comme positif, combien le sont     │
│ vraiment ?"                                                         │
│                                                                     │
│ Haute precision = peu de fausses alertes                           │
│ Basse precision = beaucoup de fausses alertes                      │
└─────────────────────────────────────────────────────────────────────┘

---

## 2.4 Recall : "Parmi les vrais cas, combien ai-je trouvés ?"

### Construire l'intuition

Maintenant, placez-vous du point de vue de la **sécurité**. Il y a 10 vraies fraudes dans le système.

**Question :** Combien de ces 10 fraudes avez-vous réellement détectées ?

C'est ce que mesure le **Recall** (aussi appelé **Sensibilité** ou **Taux de Vrais Positifs**) :

```
┌─────────────────────────────────────────────────────────────────────┐
│              RECALL : Le taux de détection                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│                     Vraies FRAUDES (réalité)                        │
│                   ┌─────────────────────────┐                       │
│                   │  TP = 7    │  FN = 3    │                       │
│                   │ (détectées)│ (manquées!)│                       │
│                   └─────────────────────────┘                       │
│                            Total = 10                               │
│                                                                     │
│   Recall = TP / (TP + FN) = 7 / 10 = 70%                           │
│                                                                     │
│   → "J'ai détecté 70% des vraies fraudes"                          │
│   → "J'ai MANQUÉ 30% des fraudes !" ⚠️                             │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
from sklearn.metrics import recall_score

recall = recall_score(y_vrai, y_pred)
recall_manuel = TP / (TP + FN)

print("📊 RECALL (Sensibilité)")
print("=" * 50)
print(f"\nFormule : TP / (TP + FN)")
print(f"        = {TP} / ({TP} + {FN})")
print(f"        = {recall:.1%}")
print(f"\n💡 Interprétation :")
print(f"   Sur 10 vraies fraudes, {TP} ont été détectées.")
print(f"   → {FN} fraudes ont été MANQUÉES ! ⚠️")

---

┌─────────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : Recall (Rappel / Sensibilité)                       │
│                                                                     │
│ Le Recall mesure la proportion de vrais positifs qui ont été       │
│ correctement identifiés par le modèle.                              │
│                                                                     │
│ Formule : Recall = TP / (TP + FN)                                  │
│                                                                     │
│ Question qu'elle répond :                                           │
│ "Parmi tous les vrais positifs, combien en ai-je trouvés ?"       │
│                                                                     │
│ Haut recall = peu de cas manqués                                   │
│ Bas recall = beaucoup de cas manqués                               │
│                                                                     │
│ Synonymes : Sensibilité, Taux de Vrais Positifs (TPR)              │
└─────────────────────────────────────────────────────────────────────┘

---

## 2.5 Le Compromis Precision-Recall

### Le dilemme

Precision et Recall sont souvent en **tension** :

```
┌─────────────────────────────────────────────────────────────────────┐
│           LE COMPROMIS PRECISION vs RECALL                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Si vous voulez HAUTE PRECISION (peu de fausses alertes) :        │
│   → Vous devenez plus "prudent" dans vos prédictions positives     │
│   → Vous ne signalez que les cas très sûrs                         │
│   → CONSÉQUENCE : Vous manquez des vrais cas (recall ↓)            │
│                                                                     │
│   ─────────────────────────────────────────────────────────────────│
│                                                                     │
│   Si vous voulez HAUT RECALL (ne manquer aucun cas) :              │
│   → Vous devenez plus "alarmiste"                                  │
│   → Vous signalez au moindre doute                                 │
│   → CONSÉQUENCE : Beaucoup de fausses alertes (precision ↓)        │
│                                                                     │
│   ─────────────────────────────────────────────────────────────────│
│                                                                     │
│   PRECISION ←─────────── ÉQUILIBRE ───────────→ RECALL             │
│   "Ne crier au loup                   "Ne manquer                  │
│    que si sûr"                         aucun loup"                 │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

**Question :** Dans quel contexte voudriez-vous maximiser le Recall, même au détriment de la Precision ?

*(Réponse attendue : Diagnostic de cancer, détection de menaces de sécurité critique — tout contexte où manquer un cas positif est catastrophique.)*

In [ ]:
# Visualiser le compromis
print("📊 Notre modèle de détection de fraude")
print("=" * 50)
print(f"\nPrecision : {precision:.1%} (fiabilité des alertes)")
print(f"Recall    : {recall:.1%} (taux de détection)")
print(f"\n💡 Analyse :")
print(f"   • Quand on alerte, on a raison {precision:.0%} du temps")
print(f"   • Mais on ne détecte que {recall:.0%} des fraudes")
print(f"   • {FN} fraudes sur 10 passent inaperçues !")

---

## 2.6 F1-Score : L'Équilibre entre Precision et Recall

### Le problème

Comparer deux modèles avec des Precision/Recall différents est compliqué :

| Modèle | Precision | Recall |
|--------|-----------|--------|
| A | 90% | 50% |
| B | 60% | 80% |

Lequel est meilleur ? Ça dépend du contexte... Mais si on veut un **score unique** qui équilibre les deux ?

### L'idée : la moyenne harmonique

Le F1-Score est la **moyenne harmonique** de Precision et Recall :

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

**Pourquoi harmonique et pas arithmétique ?**

```
┌─────────────────────────────────────────────────────────────────────┐
│     MOYENNE ARITHMÉTIQUE vs HARMONIQUE                              │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Precision = 100%, Recall = 0%                                     │
│                                                                     │
│   Moyenne arithmétique : (100 + 0) / 2 = 50%  ← Semble correct     │
│   Moyenne harmonique   : 2 × (100×0)/(100+0) = 0%  ← Plus sévère   │
│                                                                     │
│   La moyenne harmonique PÉNALISE fortement si l'une des deux       │
│   métriques est très basse. Un modèle avec Recall = 0 est inutile, │
│   même avec Precision parfaite → F1 = 0 reflète cela.              │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
from sklearn.metrics import f1_score

f1 = f1_score(y_vrai, y_pred)
f1_manuel = 2 * (precision * recall) / (precision + recall)

print("📊 F1-SCORE")
print("=" * 50)
print(f"\nFormule : 2 × (Precision × Recall) / (Precision + Recall)")
print(f"        = 2 × ({precision:.3f} × {recall:.3f}) / ({precision:.3f} + {recall:.3f})")
print(f"        = {f1:.1%}")
print(f"\n💡 Interprétation :")
print(f"   F1 = {f1:.1%} est un compromis entre :")
print(f"   • Precision = {precision:.1%}")
print(f"   • Recall = {recall:.1%}")

---

┌─────────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : F1-Score                                            │
│                                                                     │
│ Le F1-Score est la moyenne harmonique de Precision et Recall.      │
│ Il fournit un score unique qui équilibre les deux métriques.       │
│                                                                     │
│ Formule : F1 = 2 × (Precision × Recall) / (Precision + Recall)     │
│                                                                     │
│ Caractéristiques :                                                  │
│ • Valeur entre 0 et 1 (plus haut = meilleur)                       │
│ • F1 = 1 uniquement si Precision ET Recall = 1                     │
│ • F1 proche de 0 si Precision OU Recall proche de 0                │
│                                                                     │
│ Variantes :                                                         │
│ • F2-Score : donne plus de poids au Recall (médical)               │
│ • F0.5-Score : donne plus de poids à la Precision (spam)           │
└─────────────────────────────────────────────────────────────────────┘

---

## 2.7 Le Rapport de Classification Complet

Scikit-learn fournit un rapport complet avec `classification_report` :

In [ ]:
from sklearn.metrics import classification_report

print("📊 Rapport de Classification Complet")
print("=" * 55)
print(classification_report(y_vrai, y_pred, target_names=['Légitime', 'Fraude']))

### Comment lire ce rapport ?

```
┌─────────────────────────────────────────────────────────────────────┐
│     ANATOMIE DU CLASSIFICATION REPORT                               │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│              precision    recall  f1-score   support                │
│                                                                     │
│   Légitime       0.97      0.94      0.96        90   ← Classe 0    │
│   Fraude         0.58      0.70      0.64        10   ← Classe 1    │
│                                                                     │
│   ────────────────────────────────────────────────────              │
│                                                                     │
│   accuracy                           0.92       100   ← Global      │
│   macro avg      0.78      0.82      0.80       100   ← Moyenne     │
│   weighted avg   0.93      0.92      0.92       100   ← Pondérée    │
│                                                                     │
│   ────────────────────────────────────────────────────              │
│                                                                     │
│   support = nombre d'exemples dans chaque classe                    │
│   macro avg = moyenne simple (traite toutes les classes également) │
│   weighted avg = moyenne pondérée par le support                    │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

<details>
<summary>🤔 Question Socratique : Pourquoi les métriques pour "Légitime" sont-elles bien meilleures que pour "Fraude" ?</summary>

### 🔑 Réponse

C'est le **problème du déséquilibre des classes** :

1. **Plus d'exemples d'entraînement** pour la classe majoritaire → le modèle apprend mieux à reconnaître les légitimes

2. **Optimisation biaisée** : Si le modèle maximise l'accuracy globale, il favorise la classe majoritaire

3. **Moins de variations** à apprendre pour les légitimes → plus facile à classifier

**Solutions possibles :**
- Rééquilibrage des données (oversampling, undersampling, SMOTE)
- Ajustement du seuil de classification
- Utilisation de métriques adaptées (F1, ROC-AUC)
- Pondération des classes dans l'entraînement

</details>

---

## 2.8 ROC-AUC : Évaluer à Tous les Seuils

### Le problème du seuil

Un modèle de classification ne prédit pas directement "Fraude" ou "Légitime". Il prédit une **probabilité** :

```
Transaction X → Modèle → P(Fraude) = 0.73
```

C'est le **seuil** qui décide de la classification finale :
- Seuil = 0.5 (défaut) : Si P(Fraude) > 0.5 → "Fraude"
- Seuil = 0.3 (prudent) : Si P(Fraude) > 0.3 → "Fraude" (plus d'alertes, plus de recall)
- Seuil = 0.8 (strict) : Si P(Fraude) > 0.8 → "Fraude" (moins d'alertes, plus de precision)

**Question :** Comment évaluer un modèle indépendamment du seuil choisi ?

### La courbe ROC

La courbe **ROC** (Receiver Operating Characteristic) trace le Taux de Vrais Positifs (Recall) contre le Taux de Faux Positifs pour **tous les seuils possibles** :

```
┌─────────────────────────────────────────────────────────────────────┐
│              LA COURBE ROC                                          │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   Taux de Vrais Positifs (Recall)                                   │
│   1.0 ┤                              ●───────────────────           │
│       │                          ●                                  │
│       │                      ●    ← Courbe ROC du modèle           │
│   0.5 ┤                  ●                                          │
│       │              ●         ← Meilleur modèle = plus vers       │
│       │          ●               le coin supérieur gauche          │
│       │      ●                                                      │
│   0.0 ┼──●───────────────────────────────────────────────────       │
│       0.0                    0.5                         1.0        │
│               Taux de Faux Positifs (FPR)                           │
│                                                                     │
│   • Diagonale = modèle aléatoire (aussi bon que le hasard)         │
│   • Coin supérieur gauche = modèle parfait                         │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, RocCurveDisplay

# Créer un dataset plus réaliste pour la démo ROC
np.random.seed(42)
n = 1000

# Features
X = np.random.randn(n, 5)

# Target déséquilibrée (10% de positifs)
y = (X[:, 0] + X[:, 1] + np.random.randn(n) * 0.5 > 1).astype(int)

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Entraîner un modèle
model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

# Obtenir les probabilités
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Modèle entraîné. Distribution des classes de test :")
print(f"  Classe 0 : {sum(y_test == 0)}")
print(f"  Classe 1 : {sum(y_test == 1)}")

In [ ]:
# Tracer la courbe ROC
fig, ax = plt.subplots(figsize=(8, 6))

# Courbe ROC
RocCurveDisplay.from_predictions(y_test, y_proba, ax=ax, name='Logistic Regression')

# Ligne de référence (modèle aléatoire)
ax.plot([0, 1], [0, 1], 'k--', label='Modèle aléatoire')

ax.set_title('Courbe ROC')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

### L'AUC : Area Under the Curve

L'**AUC** (Area Under the ROC Curve) résume la courbe ROC en un seul chiffre :

In [ ]:
auc = roc_auc_score(y_test, y_proba)

print("📊 ROC-AUC Score")
print("=" * 50)
print(f"\nAUC = {auc:.4f}")
print(f"\n💡 Interprétation :")
print(f"   AUC = {auc:.2f} signifie que si on prend au hasard :")
print(f"   - un exemple positif")
print(f"   - un exemple négatif")
print(f"   Le modèle attribuera un score plus élevé au positif")
print(f"   dans {auc:.0%} des cas.")

In [ ]:
print("""
┌─────────────────────────────────────────────────────────────────────┐
│              INTERPRÉTATION DE L'AUC                                │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   AUC = 1.00     Modèle parfait                                     │
│   AUC > 0.90     Excellent                                          │
│   AUC > 0.80     Bon                                                │
│   AUC > 0.70     Acceptable                                         │
│   AUC = 0.50     Modèle aléatoire (inutile)                        │
│   AUC < 0.50     Pire que le hasard ! (inverser les prédictions)   │
│                                                                     │
│   AVANTAGES de l'AUC :                                              │
│   • Indépendant du seuil                                            │
│   • Robuste au déséquilibre des classes                            │
│   • Facile à comparer entre modèles                                 │
│                                                                     │
└─────────────────────────────────────────────────────────────────────┘
""")

---

┌─────────────────────────────────────────────────────────────────────┐
│ 📖 DÉFINITION : ROC-AUC                                             │
│                                                                     │
│ **ROC** (Receiver Operating Characteristic) : courbe qui trace     │
│ le Taux de Vrais Positifs (Recall) contre le Taux de Faux          │
│ Positifs pour tous les seuils de classification possibles.         │
│                                                                     │
│ **AUC** (Area Under the Curve) : aire sous la courbe ROC.          │
│ Mesure la capacité du modèle à discriminer entre classes.          │
│                                                                     │
│ Interprétation probabiliste :                                       │
│ AUC = P(score positif > score négatif pour une paire aléatoire)    │
│                                                                     │
│ Avantages :                                                         │
│ • Invariant au seuil de classification                             │
│ • Robuste au déséquilibre des classes                              │
│ • Comparable entre différents modèles et datasets                  │
└─────────────────────────────────────────────────────────────────────┘

---

## 2.9 Guide de Sélection des Métriques

```
┌─────────────────────────────────────────────────────────────────────┐
│        QUELLE MÉTRIQUE POUR QUEL CONTEXTE ?                         │
├─────────────────────────────────────────────────────────────────────┤
│                                                                     │
│   CONTEXTE                        │ MÉTRIQUE PRIORITAIRE            │
│   ────────────────────────────────┼────────────────────────────────│
│                                   │                                 │
│   Classes équilibrées             │ Accuracy, F1                    │
│   (50/50 ou proche)               │                                 │
│                                   │                                 │
│   Classes déséquilibrées          │ F1, ROC-AUC, Precision, Recall │
│   (90/10, 99/1...)                │ (pas l'accuracy !)             │
│                                   │                                 │
│   Coût élevé des FN               │ RECALL                          │
│   (cancer, fraude, sécurité)      │ "Ne manquer aucun cas"         │
│                                   │                                 │
│   Coût élevé des FP               │ PRECISION                       │
│   (spam, recommandations)         │ "Ne pas déranger inutilement"  │
│                                   │                                 │
│   Équilibre FP/FN                 │ F1-Score                        │
│                                   │                                 │
│   Comparaison de modèles          │ ROC-AUC                         │
│   (indépendant du seuil)          │                                 │
│                                   │                                 │
└─────────────────────────────────────────────────────────────────────┘
```

In [ ]:
# Fonction utilitaire pour évaluer un classificateur
def evaluer_classification(y_vrai, y_pred, y_proba=None, nom_modele="Modèle"):
    """Affiche toutes les métriques de classification importantes."""
    from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                                 f1_score, roc_auc_score, classification_report)
    
    print(f"\n📊 Évaluation : {nom_modele}")
    print("=" * 55)
    
    print(f"\nAccuracy  : {accuracy_score(y_vrai, y_pred):.4f}")
    print(f"Precision : {precision_score(y_vrai, y_pred):.4f}")
    print(f"Recall    : {recall_score(y_vrai, y_pred):.4f}")
    print(f"F1-Score  : {f1_score(y_vrai, y_pred):.4f}")
    
    if y_proba is not None:
        print(f"ROC-AUC   : {roc_auc_score(y_vrai, y_proba):.4f}")
    
    print(f"\n{classification_report(y_vrai, y_pred)}")

# Test
y_pred_test = model.predict(X_test)
evaluer_classification(y_test, y_pred_test, y_proba, "Logistic Regression")

---

## 🧪 Exercice Pratique : Évaluer un Modèle de Diagnostic Médical

Vous travaillez sur un modèle de dépistage du diabète. Voici les résultats :

In [ ]:
# Données de l'exercice
# 500 patients : 450 sains, 50 diabétiques
np.random.seed(123)

# Vraies valeurs (0 = sain, 1 = diabétique)
y_vrai_med = np.array([0]*450 + [1]*50)

# Prédictions du modèle
# Le modèle détecte 35 diabétiques sur 50, avec 20 faux positifs
y_pred_med = np.array([0]*430 + [1]*20 + [0]*15 + [1]*35)

print("📊 Données : Dépistage du diabète")
print("=" * 50)
print(f"Patients totaux : {len(y_vrai_med)}")
print(f"  - Sains : {sum(y_vrai_med == 0)}")
print(f"  - Diabétiques : {sum(y_vrai_med == 1)}")

### Votre mission :

1. Calculez la matrice de confusion
2. Calculez Accuracy, Precision, Recall, F1
3. Répondez : Ce modèle est-il satisfaisant pour un dépistage médical ?
4. Quelle métrique devrait être prioritaire ici et pourquoi ?

In [ ]:
# 🎯 À VOUS DE JOUER !

# Étape 1 : Matrice de confusion
# ...

# Étape 2 : Métriques
# ...

# Étape 3 & 4 : Analyse
# ...

### 🔑 Solution

In [ ]:
# Solution - Étape 1 : Matrice de confusion
cm_med = confusion_matrix(y_vrai_med, y_pred_med)
TN_m, FP_m, FN_m, TP_m = cm_med.ravel()

print("📊 Matrice de Confusion - Dépistage Diabète")
print("=" * 50)
print(f"\nVrais Négatifs  (TN) : {TN_m} patients sains correctement identifiés")
print(f"Faux Positifs   (FP) : {FP_m} patients sains déclarés diabétiques (stress inutile)")
print(f"Faux Négatifs   (FN) : {FN_m} diabétiques MANQUÉS ! ⚠️ (danger !)")
print(f"Vrais Positifs  (TP) : {TP_m} diabétiques correctement détectés")

In [ ]:
# Solution - Étape 2 : Métriques
accuracy_m = accuracy_score(y_vrai_med, y_pred_med)
precision_m = precision_score(y_vrai_med, y_pred_med)
recall_m = recall_score(y_vrai_med, y_pred_med)
f1_m = f1_score(y_vrai_med, y_pred_med)

print("\n📊 Métriques")
print("=" * 50)
print(f"Accuracy  : {accuracy_m:.1%}")
print(f"Precision : {precision_m:.1%}")
print(f"Recall    : {recall_m:.1%}")
print(f"F1-Score  : {f1_m:.1%}")

In [ ]:
# Solution - Étape 3 & 4 : Analyse
print("\n📋 Analyse Critique")
print("=" * 55)

print(f"\n1. L'accuracy de {accuracy_m:.1%} semble bonne, mais c'est TROMPEUR !")
print(f"   → Un modèle qui dit 'sain' pour tout le monde aurait {450/500:.1%} d'accuracy")

print(f"\n2. Le RECALL de {recall_m:.1%} est la métrique clé ici.")
print(f"   → On détecte seulement {TP_m} diabétiques sur 50")
print(f"   → {FN_m} patients diabétiques sont renvoyés chez eux sans diagnostic ! 😱")

print(f"\n3. Pour un dépistage médical, le RECALL devrait être > 90%")
print(f"   → Mieux vaut des faux positifs (examens complémentaires)")
print(f"   → Qu'un seul diabétique manqué (complications graves)")

print(f"\n💡 RECOMMANDATION : Ajuster le seuil pour augmenter le recall,")
print(f"   même si cela diminue la precision.")

---

## 🧠 Réflexion Métacognitive

Avant de passer à la suite :

1. **Pouvez-vous expliquer** la différence entre Precision et Recall à un collègue non-technique ?

2. **Dans quel cas** utiliseriez-vous l'AUC plutôt que le F1-Score ?

3. **Pourquoi** l'accuracy seule est-elle insuffisante pour les classes déséquilibrées ?

---

## 📝 Résumé

| Métrique | Formule | Question qu'elle répond |
|----------|---------|------------------------|
| **Accuracy** | (TP+TN) / Total | Quel % de prédictions correctes ? |
| **Precision** | TP / (TP+FP) | Parmi mes alertes, combien sont vraies ? |
| **Recall** | TP / (TP+FN) | Parmi les vrais cas, combien ai-je trouvés ? |
| **F1-Score** | 2×(P×R)/(P+R) | Équilibre Precision-Recall |
| **ROC-AUC** | Aire sous ROC | Capacité de discrimination (tous seuils) |

**Code essentiel :**
```python
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)

# Métriques
accuracy = accuracy_score(y_vrai, y_pred)
precision = precision_score(y_vrai, y_pred)
recall = recall_score(y_vrai, y_pred)
f1 = f1_score(y_vrai, y_pred)
auc = roc_auc_score(y_vrai, y_proba)  # Nécessite les probabilités

# Rapport complet
print(classification_report(y_vrai, y_pred))
```

---

## ➡️ Prochaine partie

Dans la **Partie 3 : Biais-Variance Tradeoff**, nous allons comprendre le compromis fondamental entre underfitting et overfitting.

**Question de transition :** Un modèle très simple (régression linéaire) et un modèle très complexe (arbre profond) font tous deux des erreurs. Mais font-ils le **même type** d'erreur ?